# babaorum E2E-5 — B3 avec lambda = 1/6 sur le gradient CTC entrant dans enc_det (Kaggle GPU T4)
Lot : docs/END_TO_END_E2E5_LOT.txt (amendement 1 §6 et amendement 2 §7). Entrées : (1) bundle v6 ; (2) kit E2E-5 ; (3) sortie E2E-3 run 3 (e2e3/mf et e2e3_fp_classes_run3.json, non régénérées). Ne sont affichés que le statut, le sha du rapport et le rapport des normes du pas 0.

In [ ]:
import importlib, os, subprocess, sys
REQUIRED_OUTSIDE_IMAGE = []          # paquets pip hors image ; vide pour la chaîne d'E2E-5
for pkg in REQUIRED_OUTSIDE_IMAGE:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
for mod in ('numpy', 'torch', 'PIL', 'lxml', *REQUIRED_OUTSIDE_IMAGE):
    importlib.import_module(mod)     # échoue tout de suite si l'image ne suffit pas


In [ ]:
import glob, hashlib, json, os, platform, shutil, subprocess, sys, zipfile
from importlib.metadata import version
from pathlib import Path

BUNDLE_SHA = "91402c71b789f36d5c30c6dd1d5de400ec1ae6358bc327840877c26486862948"
KIT_SHA = {"scripts/e2e5_lambda.py": "2653e6fbc1f3d8f9d680a4052598452f8b4829121369dd124619b21d8486daaf", "config/e2e5_lambda.json": "62e33fa9f8d19ae1db023e119300777ecc22fe4a4bc375b4ceae1bdafee0622f", "tests/test_e2e5_lambda.py": "be56ee15e70723c92c2596a0cd74acf2f947f73e47474c7d7085f56e1b1e8eea"}
HARD_STOP_S = 38 * 60
OUT = Path("/kaggle/working")
REPORT = OUT / "e2e5_lambda_report.json"


def sha256(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()


def one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    assert len(hits) == 1, (pattern, hits)
    return Path(hits[0])


# 0. cheap controls: GPU, Python, ps, torch and numpy versions
import torch  # noqa: E402
import numpy  # noqa: E402
assert torch.cuda.is_available(), "no GPU: E2E-5 runs on Kaggle GPU T4 only"
assert shutil.which("ps"), "ps missing"
print("torch", torch.__version__, "numpy", numpy.__version__, "gpu", torch.cuda.get_device_name(0), flush=True)

# 1. bundle v6, checked then extracted
zip_path = one("/kaggle/input/**/stage_s_bundle_v6.zip.bin")  # .bin: Kaggle unzips .zip uploads
got = sha256(zip_path)
assert got == BUNDLE_SHA, ("bundle sha mismatch", got)
BUNDLE = OUT / "bundle"
shutil.rmtree(BUNDLE, ignore_errors=True)
BUNDLE.mkdir(parents=True)
zipfile.ZipFile(zip_path).extractall(BUNDLE)

# 2. E2E-5 kit overlaid on the babaorum tree of the bundle
kit_script = one("/kaggle/input/**/e2e5_lambda.py")
KIT = kit_script.parent.parent
kit_sha = {}
for rel, want in KIT_SHA.items():
    src = KIT / rel
    assert src.exists(), ("kit file missing", rel)
    kit_sha[rel] = sha256(src)
    assert kit_sha[rel] == want, ("kit sha mismatch", rel, kit_sha[rel])
    dst = BUNDLE / "babaorum" / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
ROOT_E2E5 = BUNDLE / "babaorum"
PYTHONPATH = os.pathsep.join([str(ROOT_E2E5 / "src"), str(ROOT_E2E5 / "scripts")])

# 3. E2E-3 run 3 output: pages (not regenerated) and their report (sha reference)
mf_dirs = [Path(p) for p in glob.glob("/kaggle/input/**/e2e3/mf", recursive=True) if Path(p).is_dir()]
assert len(mf_dirs) == 1, ("e2e3/mf not found once", mf_dirs)
PAGES_TMP = mf_dirs[0].parent
RUN3 = one("/kaggle/input/**/e2e3_fp_classes_run3.json")

# 4. tests of the E2E-5 rules (cheap, before the pilot)
env = dict(os.environ, PYTHONPATH=PYTHONPATH)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_e2e5_lambda.py",
                "--deselect", "tests/test_e2e5_lambda.py::test_install_cell_of_the_builder_is_the_block_of_kaggle_install_txt",
                "--deselect", "tests/test_e2e5_lambda.py::test_notebook_first_code_cell_is_the_install_cell"],
               cwd=str(ROOT_E2E5), env=env, check=True, text=True)

# 5. manifest, then the pilot with a hard stop (report and summary only in /kaggle/working)
manifest = {"bundle_sha256": got, "kit_sha256": kit_sha, "run3_report_sha256": sha256(RUN3),
            "python": platform.python_version(), "torch": version("torch"), "numpy": version("numpy")}
(OUT / "e2e5_kaggle_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2), flush=True)
try:
    proc = subprocess.run([sys.executable, "scripts/e2e5_lambda.py", "--confirmed-go", "--device", "cuda",
                           "--tmp", str(PAGES_TMP), "--run3-report", str(RUN3), "--out-dir", str(OUT)],
                          cwd=str(ROOT_E2E5), env=env, capture_output=True, text=True, timeout=HARD_STOP_S)
    print(proc.stdout.strip(), flush=True)
    rc = proc.returncode
except subprocess.TimeoutExpired:
    print(json.dumps({"status": "INCOMPLETE", "report_sha256": None, "step0_ratio": None,
                      "reason": "hard stop 38 min"}), flush=True)
    rc = "timeout"
(OUT / "e2e5_run_exit.json").write_text(json.dumps({"exit_code": rc}))
